## Lecture 9 - Simplified processes (Lightning / fastai)
**Assignment: Reduce boilerplate in training loops**

Instructions:

1. Start from a plain PyTorch training loop
2. Refactor to Lightning OR implement with fastai
3. Compare code length and readability

### Task 1: Baseline PyTorch loop
Start with a plain training loop.

In [ ]:
# TODO: Build a small model and training loop in PyTorch
# This is what we do as usual!
# We load in data, then build a training loop

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Vi laddar in data och splittar i X och y
iris = load_iris()
X = iris.data
y = iris.target

# We split the data into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

# We scale our data after the split, to avoid data leakage
# We could create a Scaler, and then reuse it, but then
# we have to make sure not to run fit on both, so no data slips through
X_train = StandardScaler().fit_transform(X_train)
X_test = StandardScaler().fit_transform(X_test)

# ============ HERE WE WOULD TYPICALLY DO EDA ============ #

# we define a DL model with 3 layers (in - hidden - out)
# They have (4, 128 and 3 nodes respectively)
model = nn.Sequential(
    nn.Linear(4, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
)

# We use cross entropy loss, since we have a classification problem with >2 classes
criterion = nn.CrossEntropyLoss()

# adam is our default optimizer!
optimizer = optim.Adam(model.parameters(), lr=0.01)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model.to(device)

Sequential(
  (0): Linear(in_features=4, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=3, bias=True)
)

In [2]:
X.shape

(150, 4)

In [ ]:
# We convert our data into tensors, and package them into TensorDataset and DataLoader to facilitate training
train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long),
)

# The loader allows us to iterate over our data in batches, and also shuffle it during training
# This is also something that makes it easier for torch to handle the data, and can lead to better convergence
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

In [ ]:
# TODO: Train for a few epochs and record accuracy

epochs = 3
for _ in range(epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        # A. Zero grad: Set gradients to zero before backward pass
        optimizer.zero_grad()
    
        # B. Forward: Build the graph & get prediction
        # Outputs can often be called logits, but it is not a requirement
        outputs = model(xb)
        loss = criterion(outputs, yb)
    
        # C. Backward: AutoDiff calculates the "blame" (gradients)
        loss.backward()
        
        # D. Update: Optimizer moves weights down the hill
        optimizer.step()

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = torch.argmax(model(xb), dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
print(f"Baseline accuracy: {correct / total:.4f}")

Baseline accuracy: 0.8667


## Task 2: Refactor with Lightning OR fastai
Reduce boilerplate using a higher-level framework.